# Apply Regression with Python

*Course:* MGS 3001 WHS01 — Python Programming for Business  
*Session:* Week 14-1  
*Date & Time:* 26 May 2026, 10:00–11:15  
*Venue:* CBPM A102, Wenzhou-Kean University  
*Instructor:* Dawei Chen

### Where We Are?

`Collect ✓ → Clean ✓ → Transform ✓ → EDA ✓ → Visualize ✓ → Intro to Regression ✓` → <span style="color:blue">**Apply Regression (TODAY)**</span> → Writing (5/28)

### Learning Objectives

By the end of this session, you should be able to:

1. Run and interpret OLS regression output in Python using `statsmodels`
2. Add interaction terms and quadratic terms to a regression model
3. (Optional) Check OLS assumptions using residual plots, Q-Q plots, and VIF
4. Run multiple regression specifications systematically and export results into a clean, publication-ready table
5. Use AI effectively to generate regression code while maintaining ownership of the research design

---
## Part 1: Basic — Running a Regression in Python


Two major libraries are commonly used for statistical modeling in Python (with others including [`linearmodels`](https://pypi.org/project/linearmodels/), [`PyMC`](https://pymcmc.readthedocs.io/en/latest/README.html), [`tensoflow`](https://www.tensorflow.org/), etc.). Each serves different purposes.

| | [`statsmodels`](https://www.statsmodels.org/stable/index.html) | [`scikit-learn`](https://scikit-learn.org/stable/) |
|---|---|---|
| **Purpose** | Statistical models / tests / data exploration | Machine learning |
| **Typical output** | Coefficients, standard errors, p-values, R² | Accuracy, precision, recall, predicted values |
| **Audience** | Academic research, econometrics, social science | Data science, industry applications |
| **Example** | OLS regression with a summary table you can report in a paper | Random forest that predicts house prices |

In this course, we use `statsmodels` for demonstration.

### 1.1 Setup and Simulated Data

For this demo, we **simulate a dataset of 200 students** using the **study hours → GPA** example from last session.

In you actual project, you will typically load your own dataset. 

Recall the pipeline: `raw dataset → cleaned dataset → variable construction → final dataset for analysis (EDA, regresion, etc.)`. 

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm #check the documentation for more details
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_style('whitegrid')

### ⭐ **Bonus Question 1** 

**Before running the code below, think about this: if you wanted to simulate a dataset of 200 students with variables GPA, StudyHours, IQ, and SleepHours — how would you do it? What decisions do you need to make? (Hint: think about the range of each variable, the relationship between them, and how much randomness to add.)**

In [ ]:
# --- Simulate a student dataset (200 students) ---

np.random.seed(42)
n = 200

study_hours = np.random.normal(20, 6, n).clip(1, 35)
iq = np.random.normal(100, 15, n).clip(60, 145)
sleep_hours = np.random.normal(7, 1.2, n).clip(3, 10)

# True model: GPA depends on study hours, IQ, and sleep
gpa = (1.5 + 0.05 * study_hours + 0.02 * (iq - 100) + 0.08 * sleep_hours + np.random.normal(0, 0.3, n)).clip(1.0, 4.0)

df = pd.DataFrame({'GPA': gpa, 'StudyHours': study_hours, 'IQ': iq, 'SleepHours': sleep_hours})

print(f'Dataset: {df.shape[0]} students × {df.shape[1]} variables')
df.describe().round(3)

### 1.2 Your First Regression

Recall the regression equation from last session:

<div style="text-align: left;">

$\text{GPA} = \beta_0 + \beta_1 \times \text{StudyHours} + \varepsilon$

</div>

In Python, the workflow is three lines:
1. Define Y (dependent variable) and X (independent variables)
2. Add a constant (intercept) <span style="color:red">(May not required for other modules)</span>
3. Fit the model with `sm.OLS(Y, X).fit()`

In [ ]:
# --- Model 1: Simple Regression ---
# GPA = β₀ + β₁ × StudyHours + ε

Y = df['GPA']
X = df[['StudyHours']]
X = sm.add_constant(X) # adds the intercept (β₀)

model1 = sm.OLS(Y, X).fit() # OLS is a class, .fit() runs the regression and returns a results object - RegressionResults

print(model1.summary())
# model1.summary() 

**How to Read This Output**

Focus on four things:

| What to Look At | Where | What It Tells You |
|-----------------|-------|-------------------|
| **Coefficient (`coef`)** | Middle table | Direction and size of the effect |
| **p-value (`P>\|t\|`)** | Middle table | <span style="color:red">Is the effect statistically significant?</span>|
| **Standard Error** | Middle table | How precisely the coefficient is estimated |
| **R-squared** | Top right | How much variance does the model explain? |

### 1.3 Multiple Regression — Adding Controls

From last session: 
- Without controls, the StudyHours coefficient may absorb effects that belong to other variables (the omitted variable problem).
- Adding IQ and SleepHours gives us more credible estimates.

<div style="text-align: left;">

$\text{GPA} = \beta_0 + \beta_1 \times \text{StudyHours} + \beta_2 \times \text{IQ} + \beta_3 \times \text{SleepHours} + \varepsilon$

</div>

In [ ]:
# --- Model 2: Multiple Regression (with controls) ---
# GPA = β₀ + β₁ × StudyHours + β₂ × IQ + β₃ × SleepHours + ε

Y = df['GPA']
X = df[['StudyHours', 'IQ', 'SleepHours']] # simply add the new variables to X
X = sm.add_constant(X)

model2 = sm.OLS(Y, X).fit()

print(model2.summary())

In [ ]:
# --- Quick comparison: did the StudyHours coefficient change? ---

print('=== Coefficient on StudyHours ===')
print(f'Model 1 (no controls):   β = {model1.params["StudyHours"]:.4f}')
print(f'Model 2 (with controls): β = {model2.params["StudyHours"]:.4f}')
print()
print('=== Model Fit ===')
print(f'Model 1 — R²: {model1.rsquared:.4f},  Adj. R²: {model1.rsquared_adj:.4f}')
print(f'Model 2 — R²: {model2.rsquared:.4f},  Adj. R²: {model2.rsquared_adj:.4f}')
print()
print('Notice: the coefficient on StudyHours changes when you add controls.')
print('This is the omitted variable bias we discussed last session.')

### 1.4 Interaction Terms

We mentioned interaction and quadratic terms as advanced topics. Let's cover them now.

An **interaction term** tests whether the effect of one variable *depends on* the level of another. 

For example: does the benefit of studying more hours differ for students with higher vs. lower IQ?

<div style="text-align: left;">

$\text{GPA} = \beta_0 + \beta_1 \times \text{StudyHours} + \beta_2 \times \text{IQ} + \beta_3 \times (\text{StudyHours} \times \text{IQ}) + \varepsilon$

</div>

If $\beta_3$ is significant, the effect of study hours on GPA varies by IQ level.

In [ ]:
# --- Model 3: Interaction Term ---

# Create the interaction variable
df['StudyHours_x_IQ'] = df['StudyHours'] * df['IQ']

Y = df['GPA']
X = df[['StudyHours', 'IQ', 'SleepHours', 'StudyHours_x_IQ']]
X = sm.add_constant(X)

model3 = sm.OLS(Y, X).fit()

print(model3.summary())
print()
print('--- Interpretation ---')
p_interaction = model3.pvalues['StudyHours_x_IQ']
if p_interaction < 0.05:
    print(f'The interaction is significant (p = {p_interaction:.4f}).')
    print('The effect of study hours on GPA depends on IQ level.')
else:
    print(f'The interaction is NOT significant (p = {p_interaction:.4f}).')
    print('No evidence that the effect of study hours differs by IQ level.')

### 1.5 Quadratic Terms

A **quadratic term** tests for a non-linear (curvilinear) relationship. 

For example: does studying have diminishing returns? Maybe going from 5 to 15 hours helps a lot, but going from 25 to 35 helps less.

<div style="text-align: left;">

$\text{GPA} = \beta_0 + \beta_1 \times \text{StudyHours} + \beta_2 \times \text{StudyHours}^2 + \beta_3 \times \text{IQ} + \beta_4 \times \text{SleepHours} + \varepsilon$

</div>

If $\beta_2$ is negative and significant, the relationship curves downward — diminishing returns.

In [ ]:
# --- Model 4: Quadratic Term ---

# Create the squared variable
df['StudyHours_sq'] = df['StudyHours'] ** 2

Y = df['GPA']
X = df[['StudyHours', 'StudyHours_sq', 'IQ', 'SleepHours']]
X = sm.add_constant(X)

model4 = sm.OLS(Y, X).fit()

print(model4.summary())
print()
print('--- Interpretation ---')
p_sq = model4.pvalues['StudyHours_sq']
coef_sq = model4.params['StudyHours_sq']
if p_sq < 0.05:
    shape = 'diminishing returns (inverted U)' if coef_sq < 0 else 'accelerating returns (U-shape)'
    print(f'The quadratic term is significant (β = {coef_sq:.6f}, p = {p_sq:.4f}).')
    print(f'This suggests {shape}.')
else:
    print(f'The quadratic term is NOT significant (p = {p_sq:.4f}).')
    print('The relationship between study hours and GPA appears approximately linear.')

In [ ]:
# --- Visualize: Linear vs. Quadratic fit ---

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

x_range = np.linspace(df['StudyHours'].min(), df['StudyHours'].max(), 100)

# Linear fit (from Model 2, holding IQ and SleepHours at their means)
iq_mean = df['IQ'].mean()
sleep_mean = df['SleepHours'].mean()

y_lin = (model2.params['const']
         + model2.params['StudyHours'] * x_range
         + model2.params['IQ'] * iq_mean
         + model2.params['SleepHours'] * sleep_mean)

axes[0].scatter(df['StudyHours'], df['GPA'], alpha=0.3, s=15, color='#065A82')
axes[0].plot(x_range, y_lin, color='#E97451', linewidth=2)
axes[0].set_title('Linear Fit (Model 2)', fontsize=13)
axes[0].set_xlabel('Study Hours')
axes[0].set_ylabel('GPA')

# Quadratic fit (from Model 4)
y_quad = (model4.params['const']
          + model4.params['StudyHours'] * x_range
          + model4.params['StudyHours_sq'] * x_range**2
          + model4.params['IQ'] * iq_mean
          + model4.params['SleepHours'] * sleep_mean)

axes[1].scatter(df['StudyHours'], df['GPA'], alpha=0.3, s=15, color='#065A82')
axes[1].plot(x_range, y_quad, color='#16A34A', linewidth=2)
axes[1].set_title('Quadratic Fit (Model 4)', fontsize=13)
axes[1].set_xlabel('Study Hours')
axes[1].set_ylabel('GPA')

plt.tight_layout()
plt.show()

---
## Part 2: Advanced Topics

### 2.1 Checking OLS Assumptions

OLS regression makes several assumptions. If they are badly violated, your results may be unreliable.

| Assumption | What It Means | How to Check |
|------------|---------------|-------------|
| Linearity | The relationship between X and Y is linear | Residual plot |
| Homoscedasticity | Residuals have constant variance | Residual plot (no funnel shape) |
| Normality of residuals | Residuals are approximately normal | Q-Q plot |
| No multicollinearity | IVs are not too highly correlated | VIF |
| Independence | Observations are independent | Durbin-Watson statistic |

In [ ]:
# --- 2.1a Residual Plot (Linearity + Homoscedasticity) ---

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(model2.fittedvalues, model2.resid, alpha=0.4, s=15, color='#065A82')
ax.axhline(y=0, color='#E97451', linestyle='--', linewidth=1)
ax.set_xlabel('Fitted Values', fontsize=11)
ax.set_ylabel('Residuals', fontsize=11)
ax.set_title('Residuals vs. Fitted Values', fontsize=13)
plt.tight_layout()
plt.show()

print('✓ Random scatter around zero = good.')
print('✗ Funnel shape = heteroscedasticity. Curves = non-linearity.')

In [ ]:
# --- 2.1b Q-Q Plot (Normality of Residuals) ---

fig, ax = plt.subplots(figsize=(5, 5))
sm.qqplot(model2.resid, line='45', ax=ax, markersize=4, color='#1C7293')
ax.set_title('Q-Q Plot of Residuals', fontsize=13)
plt.tight_layout()
plt.show()

print('Points near the diagonal = approximately normal.')
print('Deviations at the tails are common and usually acceptable with large N.')

In [ ]:
# --- 2.1c VIF (Multicollinearity) ---

from statsmodels.stats.outliers_influence import variance_inflation_factor

X_vif = df[['StudyHours', 'IQ', 'SleepHours']]
X_vif = sm.add_constant(X_vif)

vif_data = pd.DataFrame({
    'Variable': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})

print(vif_data[vif_data['Variable'] != 'const'].to_string(index=False))
print()
print('Rule of thumb: VIF < 5 is acceptable. VIF > 10 is a red flag.')

In [ ]:
# --- 2.1d Robust Standard Errors ---
# If heteroscedasticity is present, use robust SEs (HC1).
# Adjusts standard errors and p-values without changing coefficients.

model2_robust = model2.get_robustcov_results(cov_type='HC1')

print(model2_robust.summary())
print()
print('Coefficients are the same. Only standard errors and p-values may change.')
print('In your paper: "We report heteroscedasticity-robust standard errors (HC1)."')

### 2.2 Running Many Regressions Systematically

In empirical research, you typically present multiple models in one table — starting from a simple baseline and progressively adding variables (i.e. stepwise regression). This demonstrates that your main finding is robust.

<span style="color:red">p-hacking:</span> `Researchers may try multiple model specifications in an attempt to obtain statistically significant results.`

Recall last session's regression table:

| Model | Variables |
|-------|----------|
| Model 1 | StudyHours only |
| Model 2 | + IQ |
| Model 3 | + SleepHours |

Let's build all three programmatically with a loop.

### ⭐ **Bonus Question 2**

Read the code above before running it. Can you answer the following?

1. What data structure is `model_specs`? What about each element inside it?
2. Why do we use a dictionary (`results = {}`) instead of a list to store the fitted models?
3. What does `spec['ivs']` return in the first iteration of the loop?
4. If you wanted to add a Model 4 that includes a new variable `Attendance`, what would you change?

In [ ]:
# --- Run multiple models in a loop ---

model_specs = [
    {'name': 'Model 1', 'ivs': ['StudyHours']},
    {'name': 'Model 2', 'ivs': ['StudyHours', 'IQ']},
    {'name': 'Model 3', 'ivs': ['StudyHours', 'IQ', 'SleepHours']},
]

results = {}    # store fitted models

for spec in model_specs:
    Y = df['GPA']
    X = sm.add_constant(df[spec['ivs']])
    fitted = sm.OLS(Y, X).fit()
    results[spec['name']] = fitted
    print(f"{spec['name']:10s}  R² = {fitted.rsquared:.4f}  Adj.R² = {fitted.rsquared_adj:.4f}")

print(f'\n{len(results)} models fitted.')

### 2.3 Exporting Regression Results

After running multiple models, the next step is to **extract the key results into a single spreadsheet** so you can:

- Compare coefficients, p-values, and R² across models side by side
- Decide whether to add, remove, or modify variables in your specification
- Identify which results to highlight in your paper

This is a working table for *you* — not yet the publication table. 

The three-line table in your paper should be created manually in Word or LaTeX once you have finalized your model specifications **(suggested)**.

In [ ]:
# --- Extract results from all models into one DataFrame ---

def extract_results(models_dict):
    """Extract key regression outputs into a flat table.
    
    Each row = one variable in one model.
    Columns: Model, Variable, Coefficient, Std Error, p-value, Significance.
    Model-level statistics (R², N) appended at the bottom.
    """
    rows = []
    for name, model in models_dict.items():
        for var in model.params.index:
            rows.append({
                'Model': name,
                'Variable': var,
                'Coefficient': round(model.params[var], 4),
                'Std Error': round(model.bse[var], 4),
                'p-value': round(model.pvalues[var], 4),
                'Significant': '***' if model.pvalues[var] < 0.001
                          else '**' if model.pvalues[var] < 0.01
                          else '*' if model.pvalues[var] < 0.05
                          else ''
            })
        # Add model-level statistics
        rows.append({
            'Model': name,
            'Variable': 'R²',
            'Coefficient': round(model.rsquared, 4),
            'Std Error': '',
            'p-value': '',
            'Significant': ''
        })
        rows.append({
            'Model': name,
            'Variable': 'Adj. R²',
            'Coefficient': round(model.rsquared_adj, 4),
            'Std Error': '',
            'p-value': '',
            'Significant': ''
        })
        rows.append({
            'Model': name,
            'Variable': 'N',
            'Coefficient': int(model.nobs),
            'Std Error': '',
            'p-value': '',
            'Significant': ''
        })
    
    return pd.DataFrame(rows)


results_df = extract_results(results)
results_df

In [ ]:
# --- Export to CSV / Excel ---
# Open in Excel or Google Sheets to compare models side by side.

results_df.to_csv('regression_results.csv', index=False)
print('Exported to: regression_results.csv')
print()
print('Next steps:')
print('1. Open the CSV in Excel or Google Sheets')
print('2. Compare coefficients and p-values across models')
print('3. Decide on your final model specification')
print('4. Manually create the three-line table in Word or LaTeX for your paper')

<span style="color:red">**Why Not Auto-Generate the Publication Table?**</span>

Packages like `stargazer` can auto-generate formatted regression tables. However, for your final paper:

- You need to **understand every number** in the table — manually building it forces you to engage with the results
- You may need to **customize labels, ordering, and notes** to fit your narrative
- Word and LaTeX give you full control over formatting that matches your paper's style

The CSV export gives you the <span style="color:red">raw material</span>. The publication table is something you craft yourself.

## Part 3: Using AI to Run Regressions

You can delegate the *coding* of regression to AI — but the *research design* is yours. This is the **4D AI Fluency Framework** applied to regression:

| Dimension | What It Means Here |
|-----------|--------------------|
| **Delegation** | You decide the research design — which Y, which X, which controls, and why. AI generates the code. |
| **Description** | Be specific in your prompt. Vague instructions → bad output. |
| **Discernment** | Check the output critically. Do signs make sense? Are p-values reasonable? |
| **Diligence** | Report results transparently. Acknowledge limitations. |

### 3.1 Bad Prompt vs. Good Prompt

**❌ Bad prompt:** *"Run a regression on my data."*

Problems: Which variable is Y? Which are X? Which are controls? Should you log-transform? Add interactions? AI cannot decide your research design for you.

**✓ Good prompt:**

```
I have a CSV file (student_data.csv) with 200 student records.
Columns: GPA, StudyHours, IQ, SleepHours.

Run three OLS regressions with GPA as the dependent variable:
- Model 1: StudyHours only
- Model 2: StudyHours + IQ
- Model 3: StudyHours + IQ + SleepHours

For each model, report coefficients with significance stars,
standard errors in parentheses, R², and N.
Format as a three-line table and export to CSV.

Also check VIF for Model 3 and produce a residual plot.
```

This prompt works because it specifies the DV, the IVs for each model, the output format, and the diagnostics.

### 3.2 A Practical AI Workflow for Regression

1. **You decide** (Delegation): DV, IVs, controls, sample, transformations
2. **You describe** (Description): Write a specific prompt with all the details above
3. **AI generates**: Code for running the models, diagnostics, and table export
4. **You check** (Discernment):
   - Do coefficient signs match your theory?
   - Are magnitudes plausible?
   - Is R² reasonable for your field? (Social science: 0.1–0.3 is common)
   - Did AI use the right variables and the right DV?
5. **You report** (Diligence): Include the full table, state limitations, no causal claims without justification

### 3.3 AI Skills for Empirical Research

- Beyond prompting general-purpose AI tools like ChatGPT or Claude, researchers are now building <span style="color:red">**structured AI skills**</span> — reusable workflows that encode methodological expertise into step-by-step instructions an AI agent can follow. 
- Think of a skill as a detailed recipe: instead of asking AI to "run a regression," a skill tells the AI exactly <span style="color:blue">what steps a well-trained researcher would take</span>, from checking data quality to running diagnostics to formatting the output.

Examples of AI skills open-sourced on Github:

1. [**Awesome Agent Skills for Empirical Research**](https://github.com/brycewang-stanford/Awesome-Agent-Skills-for-Empirical-Research): A curated collection of 23,000+ AI agent skills covering 8 social science disciplines (economics, management, political science, sociology, psychology, public health, education, finance). Organized by research workflow stage — from topic selection to journal submission. Maintained by CoPaper.AI.
2. [**Academic Research Skills for Claude Code**](https://github.com/Imbad0202/academic-research-skills): A comprehensive suite of Claude Code skills for academic research, covering the full pipeline from research to publication.

**Why does this matter for you?** 
- As you move beyond this course into research projects or graduate studies, these tools can accelerate your workflow — <span style="color:red">but only if you understand the methodology first</span>. 
- A skill automates the *steps*; it does not replace your understanding of *why* each step is necessary. That is the Discernment dimension of the 4D Framework.

---
## Apply This to Your Project

Replace the demo dataset with your own cleaned data and adapt the workflow.

This is an example — adjust the steps to fit your specific research design.

```python
# Step 1: Load your data
df = pd.read_csv('your_cleaned_data.csv')

# Step 2: Define your variables (from YOUR hypothesis)
Y = df['your_DV']                        # e.g., forks_count or ln_forks
X = df[['your_main_IV', 'control1', 'control2']]
X = sm.add_constant(X)

# Step 3: Run the model
model = sm.OLS(Y, X).fit()
print(model.summary())

# Step 4: Check assumptions (residual plot, Q-Q plot, VIF)
# Step 5: Run multiple models in a loop
# Step 6: Build the three-line table
```

---
## Summary

This session covered the full regression workflow in Python:

| Step | What You Learned |
|------|------------------|
| **Run a regression** | Define Y and X, add a constant, fit with `sm.OLS(Y, X).fit()` |
| **Add complexity** | Interaction terms test whether one effect depends on another; quadratic terms test for non-linear relationships |
| **Check assumptions** | Residual plots, Q-Q plots, VIF, and robust standard errors |
| **Run many models** | Use a loop to fit multiple specifications and compare results |
| **Export results** | Extract coefficients, p-values, and R² into a CSV for comparison; create the publication table manually |
| **Use AI wisely** | You own the research design; AI generates code; you verify the output |